In [1]:
# !# 1. Clear the problematic cache
!pip cache purge

# # 2. Install a compatible pandas version first (1.5.3 often has build issues on newer Python)
!pip install "pandas<2.0.0,>=1.5.0" --prefer-binary

# # 3. Install pytorch-forecasting without letting it re-trigger a pandas build
# !pip install pytorch-forecasting --no-deps
# !pip install lightning==2.5.0
#   # Install other missing dependencies manually if needed

!pip install pytorch_lightning
!pip install pytorch_forecasting --no-deps
!pip install https://github.com/Lightning-AI/lightning/archive/refs/heads/master.zip -U
!pip install scikit-base


Files removed: 0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 70.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  Getting requirements to build wheel ... error
error: subprocess-exited-with-error

× Getting requirements to build wheel did not run successfully.
│ exit code: 1
╰─> See above for output.

note: This error originates from a subprocess, and is likely not a problem with pip.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.1/403.1 kB 10.9 MB/s eta 0:00:00
     \ 17.2 MB 34.4 MB/s 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for lightning: filename=lightning-2.6.2-py3-none-any.whl size=850591 sha256=0580

In [2]:
import os
import gc
import random
import numpy as np
import pandas as pd
import torch
import lightning.pytorch as pl
from lightning.pytorch.callbacks import EarlyStopping
from pytorch_forecasting import TimeSeriesDataSet, TemporalFusionTransformer, QuantileLoss
from pytorch_forecasting.data import GroupNormalizer
from sklearn.metrics import mean_squared_error
import warnings
warnings.filterwarnings('ignore')

# Use GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything()

# ==========================================
# 1. MEMORY OPTIMIZATION & DATA LOADING
# ==========================================
DATA_DIR = "/kaggle/input/competitions/m5-forecasting-accuracy"

def reduce_mem_usage(df):
    numerics = ['int16', 'int32', 'int64', 'float16', 'float32', 'float64']
    for col in df.columns:
        col_type = df[col].dtypes
        if col_type in numerics:
            c_min, c_max = df[col].min(), df[col].max()
            if str(col_type)[:3] == 'int':
                if c_min > np.iinfo(np.int8).min and c_max < np.iinfo(np.int8).max:
                    df[col] = df[col].astype(np.int8)
                elif c_min > np.iinfo(np.int16).min and c_max < np.iinfo(np.int16).max:
                    df[col] = df[col].astype(np.int16)
            else:
                if c_min > np.finfo(np.float16).min and c_max < np.finfo(np.float16).max:
                    df[col] = df[col].astype(np.float16)
                elif c_min > np.finfo(np.float32).min and c_max < np.finfo(np.float32).max:
                    df[col] = df[col].astype(np.float32)
    return df

print("Loading M5 datasets...")
sales = pd.read_csv(os.path.join(DATA_DIR, "sales_train_evaluation.csv"))
calendar = pd.read_csv(os.path.join(DATA_DIR, "calendar.csv"))
sell_prices = pd.read_csv(os.path.join(DATA_DIR, "sell_prices.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

# Normalize IDs for the evaluation/validation split
validation_ids = sales['id'].str.replace('_evaluation', '_validation').unique()

# ==========================================
# 2. LONG-FORMAT MELTING & FEATURE ENGINEERING
# ==========================================
# RESEARCH TIMELINE ALIGNMENT
# To prevent RAM crashes, we don't need 5 years of history.
# We only need the last ~400 days to train a robust TFT.
START_DAY = 1500
LAST_TRAIN_DAY = 1885
VAL_END_DAY = 1913
TRUE_TEST_END = 1941

MAX_ENCODER_LENGTH = 60 # Lookback window (Matching your LGBM2 window_size)
MAX_PREDICTION_LENGTH = 28 # Forecast horizon

# Melt only the required days to save RAM
day_cols = [f"d_{i}" for i in range(START_DAY, TRUE_TEST_END + 1)]
sales_long = sales.melt(
    id_vars=['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id'],
    value_vars=day_cols,
    var_name='d',
    value_name='sales'
)

# Merge Exogenous Variables
print("Merging Calendar and Price Data...")
sales_long = sales_long.merge(calendar, on='d', how='left')
sales_long = sales_long.merge(sell_prices, on=['store_id', 'item_id', 'wm_yr_wk'], how='left')

# TFT REQUIRES an integer time index. 'd_1500' -> 1500
sales_long['time_idx'] = sales_long['d'].apply(lambda x: int(x.split('_')[1]))

# Fill missing prices with the item's median price
sales_long['sell_price'] = sales_long.groupby('id')['sell_price'].transform(lambda x: x.fillna(x.median()))

# Clean up Categoricals for PyTorch Embeddings
cat_cols = ['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id',
            'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2']
for col in cat_cols:
    sales_long[col] = sales_long[col].fillna("none").astype(str)

sales_long = reduce_mem_usage(sales_long)

# Convert 'sales' to float type for the GroupNormalizer
sales_long['sales'] = sales_long['sales'].astype(np.float32)

gc.collect()

# ==========================================
# 3. PYTORCH FORECASTING DATASETS
# ==========================================
print("\nBuilding TimeSeriesDataSets...")

# Train data: Strictly up to Day 1885
train_df = sales_long[sales_long['time_idx'] <= LAST_TRAIN_DAY]

training = TimeSeriesDataSet(
    train_df,
    time_idx="time_idx",
    target="sales",
    group_ids=["id"],
    min_encoder_length=MAX_ENCODER_LENGTH,
    max_encoder_length=MAX_ENCODER_LENGTH,
    min_prediction_length=MAX_PREDICTION_LENGTH,
    max_prediction_length=MAX_PREDICTION_LENGTH,

    # Static Variables (Things that never change)
    static_categoricals=["item_id", "dept_id", "cat_id", "store_id", "state_id"],

    # Dynamic Known Futures (Calendar events, SNAP days)
    time_varying_known_categoricals=["event_name_1", "event_type_1", "event_name_2", "event_type_2"],
    time_varying_known_reals=["time_idx", "sell_price", "snap_CA", "snap_TX", "snap_WI"],

    # Dynamic Unknowns (Historical sales)
    time_varying_unknown_reals=["sales"],

    target_normalizer=GroupNormalizer(groups=["id"], transformation="softplus"),
    add_relative_time_idx=True,
    add_target_scales=True,
    add_encoder_length=True,
)

# Batching - Drop size if CUDA goes Out Of Memory
BATCH_SIZE = 1024
train_dataloader = training.to_dataloader(train=True, batch_size=BATCH_SIZE, num_workers=4,pin_memory=True, persistent_workers=True)

# ==========================================
# 4. TRAINING THE TFT
# ==========================================
print("\n--- Commencing TFT Training ---")
tft = TemporalFusionTransformer.from_dataset(
    training,
    learning_rate=0.03,
    hidden_size=32,
    attention_head_size=4,
    dropout=0.1,
    hidden_continuous_size=16,
    loss=QuantileLoss(quantiles=[0.1, 0.5, 0.9]), # We will extract P50 for RMSE
    log_interval=10,
    reduce_on_plateau_patience=4,
)

trainer = pl.Trainer(
    max_epochs=3, # Keep low for Kaggle kernel limits
    accelerator="gpu" if torch.cuda.is_available() else "cpu",
    devices=1,
    gradient_clip_val=0.1,
    
)
trainer.fit(
    tft,
    train_dataloaders=train_dataloader,
)

# ==========================================
# 5. VALIDATION INFERENCE & ITEM-LEVEL RMSE
# ==========================================
print("\n--- Calculating Validation Item-Level Metrics ---")

# We need the history (1826-1885) to predict the validation window (1886-1913)
val_df = sales_long[(sales_long['time_idx'] > LAST_TRAIN_DAY - MAX_ENCODER_LENGTH) &
                    (sales_long['time_idx'] <= VAL_END_DAY)]

val_dataset = TimeSeriesDataSet.from_dataset(training, val_df, predict=True, stop_randomization=True)
val_dataloader = val_dataset.to_dataloader(train=False, batch_size=BATCH_SIZE * 2, num_workers=4)

# Predict and extract P50 (Median)
val_predictions = tft.predict(val_dataloader, mode="quantiles")
p50_val_forecasts = val_predictions[:, :, 1].numpy()

# Extract actuals
actual_val_sales = torch.cat([y[0] for x, y in iter(val_dataloader)]).numpy()

item_results = []
# Ensure mapping is correct based on dataloader order
unique_ids = val_dataset.decoded_index['id'].values

for idx in range(len(unique_ids)):
    mse = mean_squared_error(actual_val_sales[idx], p50_val_forecasts[idx])
    # Standardize ID back to _validation
    clean_id = unique_ids[idx].replace('_evaluation', '_validation')
    item_results.append({'id': clean_id, 'tft_rmse': np.sqrt(mse)})

item_rmse_df = pd.DataFrame(item_results)
item_rmse_df.to_csv('tft_item_level_rmse.csv', index=False)
print(f"Successfully saved item-level RMSE to 'tft_item_level_rmse.csv'")

global_rmse = np.sqrt(mean_squared_error(actual_val_sales.flatten(), p50_val_forecasts.flatten()))
print(f"Our final global TFT val rmse score is {global_rmse:.4f}")

# ==========================================
# 6. KAGGLE SUBMISSION INFERENCE
# ==========================================
print("\n--- Generating Kaggle Submission (Days 1914-1941) ---")

# We need the history (1854-1913) to predict the true future (1914-1941)
test_df = sales_long[sales_long['time_idx'] > VAL_END_DAY - MAX_ENCODER_LENGTH]

test_dataset = TimeSeriesDataSet.from_dataset(training, test_df, predict=True, stop_randomization=True)
test_dataloader = test_dataset.to_dataloader(train=False, batch_size=BATCH_SIZE * 2, num_workers=2)

test_predictions = tft.predict(test_dataloader, mode="quantiles")
p50_test_forecasts = test_predictions[:, :, 1].numpy()
test_ids = test_dataset.decoded_index['id'].values

# Format for Kaggle
f_cols = [f"F{i}" for i in range(1, MAX_PREDICTION_LENGTH + 1)]
sub_val = pd.DataFrame(p50_test_forecasts, columns=f_cols)
sub_val.insert(0, 'id', [i.replace('_evaluation', '_validation') for i in test_ids])

sub_eval = sub_val.copy()
sub_eval['id'] = sub_eval['id'].str.replace('_validation', '_evaluation')

final_submission = pd.concat([sub_val, sub_eval], ignore_index=True)
final_submission.to_csv("tft_final_submission.csv", index=False)
print("Successfully exported 'tft_final_submission.csv'!")

Using device: cuda
Loading M5 datasets...
Merging Calendar and Price Data...

Building TimeSeriesDataSets...

--- Commencing TFT Training ---


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
2026-05-01 11:41:40.980374: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777635701.221304      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777635701.279852      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempti

┏━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃    ┃ Name                               ┃ Type                            ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0  │ loss                               │ QuantileLoss                    │      0 │ train │     0 │
│ 1  │ logging_metrics                    │ ModuleList                      │      0 │ train │     0 │
│ 2  │ input_embeddings                   │ MultiEmbedding                  │ 98.0 K │ train │     0 │
│ 3  │ prescalers                         │ ModuleDict                      │    320 │ train │     0 │
│ 4  │ static_variable_selection          │ VariableSelectionNetwork        │  7.0 K │ train │     0 │
│ 5  │ encoder_variable_selection         │ VariableSelectionNetwork        │ 15.4 K │ train │     0 │
│ 6  │ decoder_variable_selection         │ VariableSelectionNetwork        │ 13.2 K │ train │     0 │
│ 7  │ static_context_variable_selection  │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 8  │ static_context_initial_hidden_lstm │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 9  │ static_context_initial_cell_lstm   │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 10 │ static_context_enrichment          │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 11 │ lstm_encoder                       │ LSTM                            │  8.4 K │ train │     0 │
│ 12 │ lstm_decoder                       │ LSTM                            │  8.4 K │ train │     0 │
│ 13 │ post_lstm_gate_encoder             │ GatedLinearUnit                 │  2.1 K │ train │     0 │
│ 14 │ post_lstm_add_norm_encoder         │ AddNorm                         │     64 │ train │     0 │
│ 15 │ static_enrichment                  │ GatedResidualNetwork            │  5.3 K │ train │     0 │
│ 16 │ multihead_attn                     │ InterpretableMultiHeadAttention │  2.6 K │ train │     0 │
│ 17 │ post_attn_gate_norm                │ GateAddNorm                     │  2.2 K │ train │     0 │
│ 18 │ pos_wise_ff                        │ GatedResidualNetwork            │  4.3 K │ train │     0 │
│ 19 │ pre_output_gate_norm               │ GateAddNorm                     │  2.2 K │ train │     0 │
│ 20 │ output_layer                       │ Linear                          │     99 │ train │     0 │
└────┴────────────────────────────────────┴─────────────────────────────────┴────────┴───────┴───────┘

Trainable params: 186 K                                                                                            
Non-trainable params: 0                                                                                            
Total params: 186 K                                                                                                
Total estimated model params size (MB): 0.745                                                                      
Modules in train mode: 455                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()